In [1]:
from itertools import chain
import json
from monty.json import MontyDecoder
import plotly.graph_objects as go
import plotly.express as px
import os
from pymatgen.core import Element


from utils_kga.statistical_analysis.get_spin_and_bond_angle_statistics import *
from utils_kga.general import pretty_plot

In [ ]:
# Load edge-df
# replace to change between analyzing either cryst. uniques or all structures
with open("data/dfs_of_magnetic_edge_information.json") as f:
# with open("data/dfs_of_magnetic_edge_information_include_crystallographic_multiples.json") as f: 
    dict_all_stats = json.load(f)
all_stats = {key: pd.DataFrame.from_dict(df) for key, df in dict_all_stats.items()}

# For metadata filtering and computation of occurrences in magnetic primitive cells
with open("../../data_retrieval_and_preprocessing_MAGNDATA/data/df_grouped_and_chosen_commensurate_MAGNDATA.json") as f:
    df = json.load(f, cls=MontyDecoder)

plot_dir = "plots/TM_octahedra_selected_d3-d3_and_hs-d5-d5_nearest-neighbors"
os.makedirs(plot_dir, exist_ok=True)

In [3]:
# Add is_tm bool for later easier analysis
for ang_df in all_stats.values():
    ang_df["site_is_tm"] = ang_df["site_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["site_to_is_tm"] = ang_df["site_to_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["ligand_el_set"] = ang_df["ligand_elements"].apply(lambda ls: set(ls))

    ang_df["site_ion"] = ang_df.apply(lambda r: r["site_element"] + str(int(r["site_oxidation"])) + "+", axis=1)
    ang_df["site_to_ion"] = ang_df.apply(lambda r: r["site_to_element"] + str(int(r["site_to_oxidation"])) + "+", axis=1)

In [4]:
d3_ions = ["Cr3+"]
d5_ions = ["Fe3+", "Mn2+"] 

In [5]:
# d3-d3 case (Cr3+ in TMoct)
ligand_multiplicity_bool = False
ligand_multiplicity_string = "no ligand multiplicity included"
normalize_bool = False
normalize_string = "absolute occurrences"

for data_string in ["all edges w. selected d3-d3 TM octahedra at both nodes",
                    "all oxygen edges w. selected d3-d3 TM octahedra at both nodes",]:
    ligand_tracker = []
    print("--------------------------------------------------------------------")
    print(normalize_string, ligand_multiplicity_string, data_string)
    print("--------------------------------------------------------------------")
    entries = 0
    almost_collinear_entries = 0
    all_spin_occus = []
    for md_id, ang_df in all_stats.items():
        subdf = ang_df.loc[(ang_df["site_ce"]=="O:6")
                & (ang_df["site_to_ce"]=="O:6")
                & (ang_df["site_is_tm"])
                & (ang_df["site_to_is_tm"])
        ]
        if "oxygen" in data_string:
            subdf = subdf.loc[subdf["ligand_el_set"]=={"O"}]

        subdf = subdf.loc[(subdf["site_ion"].isin(d3_ions)) & (subdf["site_to_ion"].isin(d3_ions))]

        if not subdf.empty:
            ligand_tracker.append("-".join(set(list(chain.from_iterable(subdf["ligand_elements"].values)))))
            entries += 1
            n_lattice_points = df.at[md_id, "n_lattice_points"]
            occus = get_bond_angle_occurrences(df=subdf,
                                                                include_ligand_multiplicity=ligand_multiplicity_bool,
                                                                normalize=normalize_bool,
                                                                n_lattice_points=n_lattice_points,
                                                                spin_angle_round=0,
                                                                bond_angle_round=7)
            all_spin_occus.extend(occus)
            if any([True for e in occus if (e[0] >= 170.0 or e[0] <= 10.0)]):
                almost_collinear_entries += 1
    all_spin_occus_df = pd.DataFrame(columns=["spin_angle", "bond_angle", "occurrence"], data=all_spin_occus)
    print(Counter(ligand_tracker))

    # Count percentage of almost collinear orderings with bond angles around 90+-15° that are AFM and that are FM
    afm_l = all_spin_occus_df.loc[(all_spin_occus_df["bond_angle"] <= 105.0) & (all_spin_occus_df["bond_angle"] >= 75.0)
                                    &(all_spin_occus_df["spin_angle"] >= 170.0) ]["occurrence"].values.sum()
    fm_l = all_spin_occus_df.loc[(all_spin_occus_df["bond_angle"] <= 105.0) & (all_spin_occus_df["bond_angle"] >= 75.0)
                                    &(all_spin_occus_df["spin_angle"] <= 10.0) ]["occurrence"].values.sum()
    print(f"Fraction of collinear orderings in (90+-15)° that are AFM: {afm_l / (afm_l + fm_l)}")
    print(f"Fraction of collinear orderings in (90+-15)° that are FM: {fm_l / (afm_l + fm_l)}")

    for spin_ang_lim in [10.0, 170.0]:
        if spin_ang_lim == 10.0:
            sub_df = all_spin_occus_df.loc[all_spin_occus_df["spin_angle"] <= spin_ang_lim]
        else:
            sub_df = all_spin_occus_df.loc[all_spin_occus_df["spin_angle"] >= spin_ang_lim]

        spin_ang_lim_string = f">=_{spin_ang_lim}_spin_angle" if spin_ang_lim == 170.0 else f"<=_{spin_ang_lim}_spin_angle"

        # Print fraction of bond angles > 110°
        print(data_string,
            spin_ang_lim_string,
                "Fraction of bond angles above 110°: ",
                sub_df.loc[sub_df["bond_angle"] > 110.0]["occurrence"].values.sum() /
                sub_df.loc[sub_df["bond_angle"] > 0.0]["occurrence"].values.sum())
        # Print fraction of bond angles around 90°
        print(data_string,
                spin_ang_lim_string,
                "Fraction of bond angles between 75° and 105°: ",
                sub_df.loc[(sub_df["bond_angle"] <= 105.0) & (sub_df["bond_angle"] >= 75.0)]["occurrence"].values.sum() /
                sub_df.loc[sub_df["bond_angle"] > 0.0]["occurrence"].values.sum())

        one_d_fig = go.Figure(layout=go.Layout(xaxis=go.layout.XAxis(title="Bond angle (°)"),
                                                yaxis=go.layout.YAxis(title="Occurrence")))

        one_d_fig.add_trace(go.Histogram(
            histfunc="sum",
            x=sub_df["bond_angle"].values,
            y=sub_df["occurrence"].values,
            nbinsx=181,
            name=spin_ang_lim,
            marker_color="#025268"
        ))
        one_d_fig = pretty_plot(one_d_fig)
        one_d_fig.update_layout(title=dict(
            text=f"{data_string} ({entries} structures ({almost_collinear_entries} almost coll.), {np.sum(sub_df.occurrence.values)} / {np.sum(all_spin_occus_df.occurrence.values)} occurrences), {ligand_multiplicity_string}, {normalize_string}, spin_ang_lim = {spin_ang_lim}°",
        font=dict(size=12)))

        one_d_fig.update_yaxes(zeroline=False)
        #y_range = [0, 90] if "oxygen" in data_string else [0, 165]
        y_range = [0, 52] if "oxygen" in data_string else [0, 52]
        one_d_fig.update_layout(xaxis_range=[57, 182], yaxis_range=y_range)
        o = "oxygen" if "oxygen" in data_string else ""
        one_d_fig.write_image(os.path.join(plot_dir, f"bond_angle_distributions_{spin_ang_lim_string}_{o}connected-d3-TM-oct_{ligand_multiplicity_string.replace(' ', '_')}.pdf"))

--------------------------------------------------------------------
absolute occurrences no ligand multiplicity included all edges w. selected d3-d3 TM octahedra at both nodes
--------------------------------------------------------------------
Counter({'O': 30, 'F': 4, 'S': 4, 'N': 2, 'Si-C': 1, 'Sb': 1, 'Se': 1, 'Te': 1, 'Cl': 1})
Fraction of collinear interactions in (90+-15)° that are AFM: 0.4221879815100154
Fraction of collinear interactions in (90+-15)° that are FM: 0.5778120184899846
all edges w. selected d3-d3 TM octahedra at both nodes <=_10.0_spin_angle Fraction of bond angles above 110°:  0.30601092896174864
all edges w. selected d3-d3 TM octahedra at both nodes <=_10.0_spin_angle Fraction of bond angles between 75° and 105°:  0.6830601092896175
all edges w. selected d3-d3 TM octahedra at both nodes >=_170.0_spin_angle Fraction of bond angles above 110°:  0.7292817679558011
all edges w. selected d3-d3 TM octahedra at both nodes >=_170.0_spin_angle Fraction of bond angles be

In [6]:
# d5-d5 case
ligand_multiplicity_bool = False
ligand_multiplicity_string = "no ligand multiplicity included"
normalize_bool = False
normalize_string = "absolute occurrences"

for data_string in ["all edges with selected d5-d5 (t2g3eg2) TM octahedra at both nodes",
                    "all oxygen edges with selected d5-d5 (t2g3eg2) TM octahedra at both nodes"]:
    ligand_tracker = []
    print("--------------------------------------------------------------------")
    print(normalize_string, ligand_multiplicity_string, data_string)
    print("--------------------------------------------------------------------")
    entries = 0
    almost_collinear_entries = 0
    all_spin_occus = []
    for md_id, ang_df in all_stats.items():
        subdf = ang_df.loc[(ang_df["site_ce"]=="O:6")
                & (ang_df["site_to_ce"]=="O:6")
                & (ang_df["site_is_tm"])
                & (ang_df["site_to_is_tm"])
        ]
        if "oxygen" in data_string:
            subdf = subdf.loc[subdf["ligand_el_set"]=={"O"}]

        subdf = subdf.loc[subdf["site_ion"].isin(d5_ions) & (subdf["site_to_ion"].isin(d5_ions))]

        if not subdf.empty:
            ligand_tracker.append("-".join(set(list(chain.from_iterable(subdf["ligand_elements"].values)))))
            entries += 1
            n_lattice_points = df.at[md_id, "n_lattice_points"]
            occus = get_bond_angle_occurrences(df=subdf,
                                                                include_ligand_multiplicity=ligand_multiplicity_bool,
                                                                normalize=normalize_bool,
                                                                n_lattice_points=n_lattice_points,
                                                                spin_angle_round=0,
                                                                bond_angle_round=7)
            all_spin_occus.extend(occus)
            if any([True for e in occus if (e[0] >= 170.0 or e[0] <= 10.0)]):
                almost_collinear_entries += 1
    all_spin_occus_df = pd.DataFrame(columns=["spin_angle", "bond_angle", "occurrence"], data=all_spin_occus)
    print(Counter(ligand_tracker))

    # Count percentage of almost collinear orderings with bond angles around 90+-15° that are AFM and that are FM
    afm_l = all_spin_occus_df.loc[(all_spin_occus_df["bond_angle"] <= 105.0) & (all_spin_occus_df["bond_angle"] >= 75.0)
                                    &(all_spin_occus_df["spin_angle"] >= 170.0) ]["occurrence"].values.sum()
    fm_l = all_spin_occus_df.loc[(all_spin_occus_df["bond_angle"] <= 105.0) & (all_spin_occus_df["bond_angle"] >= 75.0)
                                    &(all_spin_occus_df["spin_angle"] <= 10.0) ]["occurrence"].values.sum()
    print(f"Fraction of collinear orderings in (90+-15)° that are AFM: {afm_l / (afm_l + fm_l)}")
    print(f"Fraction of collinear orderings in (90+-15)° that are FM: {fm_l / (afm_l + fm_l)}")

    for spin_ang_lim in [10.0, 170.0]:
        if spin_ang_lim == 10.0:
            sub_df = all_spin_occus_df.loc[all_spin_occus_df["spin_angle"] <= spin_ang_lim]
        else:
            sub_df = all_spin_occus_df.loc[all_spin_occus_df["spin_angle"] >= spin_ang_lim]

        spin_ang_lim_string = f">=_{spin_ang_lim}_spin_angle" if spin_ang_lim == 170.0 else f"<=_{spin_ang_lim}_spin_angle"

        # Print fraction of bond angles > 110°
        print(data_string,
            spin_ang_lim_string,
                "Fraction of bond angles above 110°: ",
                sub_df.loc[sub_df["bond_angle"] > 110.0]["occurrence"].values.sum() /
                sub_df.loc[sub_df["bond_angle"] > 0.0]["occurrence"].values.sum())
        # Print fraction of bond angles around 90°
        print(data_string,
                spin_ang_lim_string,
                "Fraction of bond angles between 75° and 105°: ",
                sub_df.loc[(sub_df["bond_angle"] <= 105.0) & (sub_df["bond_angle"] >= 75.0)]["occurrence"].values.sum() /
                sub_df.loc[sub_df["bond_angle"] > 0.0]["occurrence"].values.sum())

        one_d_fig = go.Figure(layout=go.Layout(xaxis=go.layout.XAxis(title="Bond angle (°)"),
                                                yaxis=go.layout.YAxis(title="Occurrence")))

        one_d_fig.add_trace(go.Histogram(
            histfunc="sum",
            x=sub_df["bond_angle"].values,
            y=sub_df["occurrence"].values,
            nbinsx=181,
            name=spin_ang_lim,
            marker_color="#025268"
        ))
        one_d_fig = pretty_plot(one_d_fig)
        one_d_fig.update_layout(title=dict(
            text=f"{data_string} ({entries} structures ({almost_collinear_entries} almost coll.), {np.sum(sub_df.occurrence.values)} / {np.sum(all_spin_occus_df.occurrence.values)} occurrences), {ligand_multiplicity_string}, {normalize_string}, spin_ang_lim = {spin_ang_lim}°",
        font=dict(size=12)))

        one_d_fig.update_yaxes(zeroline=False)
        # y_range = [0, 215] if "oxygen" in data_string else [0, 220]
        y_range = [0, 115] if "oxygen" in data_string else [0, 115]
        one_d_fig.update_layout(xaxis_range=[67, 182], yaxis_range=y_range)
        o = "oxygen" if "oxygen" in data_string else ""
        one_d_fig.write_image(os.path.join(plot_dir, f"bond_angle_distributions_{spin_ang_lim_string}_{o}connected_hs-d5-TM-oct_{ligand_multiplicity_string.replace(' ', '_')}.pdf"))

--------------------------------------------------------------------
absolute occurrences no ligand multiplicity included all edges with selected d5-d5 (t2g3eg2) TM octahedra at both nodes
--------------------------------------------------------------------
Counter({'O': 74, 'F': 16, 'S': 4, 'Te': 4, 'Se': 3, 'Br': 2, 'As': 1, 'F-O': 1, 'I': 1, 'Sb': 1, 'Cl': 1})
Fraction of collinear interactions in (90+-15)° that are AFM: 0.5688888888888889
Fraction of collinear interactions in (90+-15)° that are FM: 0.4311111111111111
all edges with selected d5-d5 (t2g3eg2) TM octahedra at both nodes <=_10.0_spin_angle Fraction of bond angles above 110°:  0.16806722689075632
all edges with selected d5-d5 (t2g3eg2) TM octahedra at both nodes <=_10.0_spin_angle Fraction of bond angles between 75° and 105°:  0.8151260504201681
all edges with selected d5-d5 (t2g3eg2) TM octahedra at both nodes >=_170.0_spin_angle Fraction of bond angles above 110°:  0.5688073394495413
all edges with selected d5-d5 (t2g3